# Strands Agent 

#### Authenticate with AWS

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()


#### Choose the set of metrics for evaluation

In [ ]:
metrics = all_metrics
print("You've chosen the following metrics for evaluating the agent deployed with AgentCore:")
for m in metrics:
    print(f"  - {m}")

#### Select AWS region and Foundation Model 

In [ ]:
region = "us-east-1"
print(f"AWS Region: {region}")


#### Install UAEF 
`pip install uaef`

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Single Agent: Weekly Weather Assistant

A weather agent with 5 tools providing 7-day forecasts (Monday–Sunday) for temperature, air quality, weather condition, humidity, and wind speed.

#### Step 1: Define tools and create the agent

In [ ]:
from strands import Agent, tool
from uaef.adapters import StrandsAdapter
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone

# 7-day weather data (Monday to Sunday)
WEEKLY_DATA = {
    "Monday":    {"temperature_f": 58, "air_quality_aqi": 42, "condition": "sunny",  "humidity_pct": 45, "wind_speed_mph": 8},
    "Tuesday":   {"temperature_f": 63, "air_quality_aqi": 55, "condition": "cloudy", "humidity_pct": 62, "wind_speed_mph": 12},
    "Wednesday": {"temperature_f": 71, "air_quality_aqi": 38, "condition": "sunny",  "humidity_pct": 40, "wind_speed_mph": 5},
    "Thursday":  {"temperature_f": 55, "air_quality_aqi": 72, "condition": "rain",   "humidity_pct": 85, "wind_speed_mph": 18},
    "Friday":    {"temperature_f": 49, "air_quality_aqi": 65, "condition": "rain",   "humidity_pct": 90, "wind_speed_mph": 22},
    "Saturday":  {"temperature_f": 67, "air_quality_aqi": 30, "condition": "sunny",  "humidity_pct": 38, "wind_speed_mph": 6},
    "Sunday":    {"temperature_f": 74, "air_quality_aqi": 28, "condition": "sunny",  "humidity_pct": 35, "wind_speed_mph": 4},
}

@tool
def get_temperature(day: str) -> dict:
    """Get the temperature forecast for a given day of the week (Monday-Sunday)."""
    day_cap = day.capitalize()
    if day_cap in WEEKLY_DATA:
        return {"day": day_cap, "temperature_f": WEEKLY_DATA[day_cap]["temperature_f"]}
    return {"error": f"No data for {day}"}

@tool
def get_air_quality(day: str) -> dict:
    """Get the air quality index (AQI) for a given day of the week (Monday-Sunday)."""
    day_cap = day.capitalize()
    if day_cap in WEEKLY_DATA:
        return {"day": day_cap, "air_quality_aqi": WEEKLY_DATA[day_cap]["air_quality_aqi"]}
    return {"error": f"No data for {day}"}

@tool
def get_weather_condition(day: str) -> dict:
    """Get the weather condition (sunny, cloudy, rain) for a given day of the week (Monday-Sunday)."""
    day_cap = day.capitalize()
    if day_cap in WEEKLY_DATA:
        return {"day": day_cap, "condition": WEEKLY_DATA[day_cap]["condition"]}
    return {"error": f"No data for {day}"}

@tool
def get_humidity(day: str) -> dict:
    """Get the humidity percentage for a given day of the week (Monday-Sunday)."""
    day_cap = day.capitalize()
    if day_cap in WEEKLY_DATA:
        return {"day": day_cap, "humidity_pct": WEEKLY_DATA[day_cap]["humidity_pct"]}
    return {"error": f"No data for {day}"}

@tool
def get_wind_speed(day: str) -> dict:
    """Get the wind speed in mph for a given day of the week (Monday-Sunday)."""
    day_cap = day.capitalize()
    if day_cap in WEEKLY_DATA:
        return {"day": day_cap, "wind_speed_mph": WEEKLY_DATA[day_cap]["wind_speed_mph"]}
    return {"error": f"No data for {day}"}

# Create agent
strands_agent = Agent(
    tools=[get_temperature, get_air_quality, get_weather_condition, get_humidity, get_wind_speed],
    system_prompt=(
        "You are a weekly weather assistant. You have access to 7-day forecast data (Monday through Sunday). "
        "Use the appropriate tool to answer questions about temperature, air quality, weather conditions, "
        "humidity, or wind speed for any day of the week. Always call the relevant tool before answering."
    ),
)
print("✓ Weather agent created with 5 tools: get_temperature, get_air_quality, get_weather_condition, get_humidity, get_wind_speed")

#### Step 2: Manual evaluation — correct and incorrect examples

We evaluate two individual queries manually:
1. A correct case where the expected answer matches the tool output
2. An incorrect case where the expected answer intentionally mismatches the tool output (to demonstrate low scores)

In [ ]:
adapter = StrandsAdapter()

# --- Case 1: Correct expected answer (Monday temp is 58°F) ---
strands_agent.messages.clear()
result_correct = strands_agent("What is the temperature on Monday?")
trace_correct = adapter.transform_to_canonical({
    "messages": strands_agent.messages,
    "metrics_summary": strands_agent.event_loop_metrics.get_summary(),
    "stop_reason": result_correct.stop_reason,
    "session_id": "manual_correct",
})
gt_correct = GroundTruth(
    expected_output="The temperature on Monday is 58°F.",
    expected_tool_calls=[ToolCall(name="get_temperature", arguments={"day": "Monday"}, timestamp=datetime.now(timezone.utc))],
)

# --- Case 2: Incorrect expected answer (Monday temp is 58°F, but we expect 95°F) ---
strands_agent.messages.clear()
result_wrong = strands_agent("What is the temperature on Monday?")
trace_wrong = adapter.transform_to_canonical({
    "messages": strands_agent.messages,
    "metrics_summary": strands_agent.event_loop_metrics.get_summary(),
    "stop_reason": result_wrong.stop_reason,
    "session_id": "manual_wrong",
})
gt_wrong = GroundTruth(
    expected_output="The temperature on Monday is 95°F.",
    expected_tool_calls=[ToolCall(name="get_temperature", arguments={"day": "Monday"}, timestamp=datetime.now(timezone.utc))],
)

# Evaluate both
eval_correct = evaluate(trace=trace_correct, ground_truth=gt_correct, metrics=single_ag_metrics)
eval_wrong = evaluate(trace=trace_wrong, ground_truth=gt_wrong, metrics=single_ag_metrics)

print(f"{'='*60}")
print("MANUAL EVALUATION — SINGLE QUERIES")
print(f"{'='*60}")
print(f"\n  ✓ Correct case (expect 58°F):  score={eval_correct.overall_score:.2f}  passed={eval_correct.passed}")
print(f"  ✗ Wrong case   (expect 95°F):  score={eval_wrong.overall_score:.2f}  passed={eval_wrong.passed}")

print(f"\nDimension breakdown (correct case):")
for dim in eval_correct.dimension_results:
    print(f"  {dim.dimension_name}: {dim.aggregate_score:.2f}")

print(f"\nDimension breakdown (wrong case):")
for dim in eval_wrong.dimension_results:
    print(f"  {dim.dimension_name}: {dim.aggregate_score:.2f}")

#### Step 3: Batch evaluation from WeatherQuestions.xlsx

Load test cases from `data/WeatherQuestions.xlsx` and run batch evaluation.

In [ ]:
import pandas as pd
import json
import uuid

# Load weather test cases from Excel
excel_path = "data/WeatherQuestions.xlsx"
df = pd.read_excel(excel_path)
test_cases = df.to_dict(orient="records")
print(f"✓ Loaded {len(test_cases)} test cases from {excel_path}")
print(f"  Columns: {list(df.columns)}")
df.head()

In [ ]:
# Run agent on each query and collect traces
traces = []
ground_truths = []

for i, tc in enumerate(test_cases):
    query = str(tc["query"])
    expected = str(tc["expected_output"])

    # Parse expected tool calls from JSON string
    expected_tools = []
    raw_tools = tc.get("expected_tool_calls")
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t["name"], arguments=t["arguments"], timestamp=datetime.now(timezone.utc)
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    strands_agent.messages.clear()
    agent_result = strands_agent(query)

    trace = adapter.transform_to_canonical({
        "messages": strands_agent.messages,
        "metrics_summary": strands_agent.event_loop_metrics.get_summary(),
        "stop_reason": agent_result.stop_reason,
        "session_id": f"weather_{i:03d}",
    })
    traces.append(trace)
    ground_truths.append(GroundTruth(expected_output=expected, expected_tool_calls=expected_tools))

    print(f"  [{i+1}/{len(test_cases)}] {query}  →  {len(trace.tool_calls)} tool call(s)")

print(f"\n✓ Collected {len(traces)} traces")

# Batch evaluate
results = batch_evaluate(traces=traces, ground_truths=ground_truths, metrics=single_ag_metrics, max_workers=4)

print(f"\n{'='*60}")
print("BATCH EVALUATION RESULTS — WEEKLY WEATHER ASSISTANT")
print(f"{'='*60}")
for i, r in enumerate(results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} [{i+1}] {test_cases[i]['query'][:50]:50s} → {r.overall_score:.2f}")

avg = sum(r.overall_score for r in results) / len(results)
pr = sum(1 for r in results if r.passed) / len(results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

# Detailed dimension breakdown for last result
print(f"\n{'='*60}")
print("DIMENSION BREAKDOWN (last query)")
for dimension in results[-1].dimension_results:
    print(f"\n  {dimension.dimension_name} ({dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "N/A"
        print(f"    {metric.metric_name}: {score_str}")

# Saving metrics

In [ ]:
# Save metric results
from uaef.utils import save_metric_results

filepath = save_metric_results(results, test_cases, prefix="strands_weather")

## Multi-Agent Strands: Single Question Evaluation

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()

import uuid
from datetime import datetime, timezone

from strands import Agent, tool
from uaef.adapters import StrandsAdapter
from uaef.api import evaluate
from uaef.models import (
    AgentTrace,
    GroundTruth,
    MultiAgentTrace,
    ToolCall,
)

### Create two Strands agents with system prompts

In [ ]:
research_agent = Agent(
    # tools=[],
    system_prompt=(
        "You are a research specialist. When given a question, provide relevant "
        "background information, key facts, and context that would help answer it. "
        "Be concise and factual. Do not provide a final answer — just gather the "
        "relevant information."
    ),
)


responder_agent = Agent(
    # tools=[synthesize_answer],
    system_prompt=(
        "You are a response specialist. You receive a user's question along with "
        "search context gathered by another agent. Use the synthesize_answer tool "
        "with the question and context, then produce a clear, well-structured answer. "
        "If the search context is insufficient or not relevant, use your own knowledge "
        "to provide an accurate and helpful answer."
    ),
)



### Multi-agent orchestration

In [ ]:
adapter = StrandsAdapter()


def run_search_then_respond(query: str, session_id: str = None) -> MultiAgentTrace:
    """
    Two-stage pipeline:
      1. research_agent retrieves relevant information
      2. responder_agent synthesizes a final answer from those results

    The adapter's multi-agent support handles trace assembly,
    coordination event inference, and workflow status automatically.
    """
    session_id = session_id or str(uuid.uuid4())

    # --- Stage 1: Web Search Agent ---
    research_agent.messages.clear()
    search_result = research_agent(f"Search for: {query}")

    # Extract the search agent's last assistant message as context
    search_context = ""
    for msg in reversed(research_agent.messages):
        if msg.get("role") == "assistant":
            content = msg.get("content", [])
            for block in content:
                if isinstance(block, dict) and "text" in block:
                    search_context = block["text"]
                    break
            if search_context:
                break
    if not search_context:
        search_context = str(search_result)

    # --- Stage 2: Responder Agent ---
    responder_agent.messages.clear()
    responder_prompt = (
        f'The user asked: "{query}"\n\n'
        f"Here is the search context gathered by the search agent:\n"
        f"{search_context}\n\n"
        f"Please synthesize a clear, accurate answer."
    )
    responder_result = responder_agent(responder_prompt)

    # --- Transform via adapter (handles MultiAgentTrace assembly) ---
    multi_trace = adapter.transform_to_canonical({
        "agents": {
            "research_agent": {
                "messages": list(research_agent.messages),
                "metrics_summary": research_agent.event_loop_metrics.get_summary(),
                "stop_reason": search_result.stop_reason,
            },
            "responder_agent": {
                "messages": list(responder_agent.messages),
                "metrics_summary": responder_agent.event_loop_metrics.get_summary(),
                "stop_reason": responder_result.stop_reason,
            },
        },
        "session_id": session_id,
    })
    return multi_trace


### Run query

In [ ]:

queries = [
    {
        "query": "What is photosynthesis?",
        "expected": ("Photosynthesis is the process by which green plants use sunlight to synthesize foods "
                     "from carbon dioxide and water."),
        "expected_tools": {
            "research_agent": [{"name": "search", "arguments": {"query": "photosynthesis"}}],
            "responder_agent": [{"name": "synthesize_answer", "arguments": {}}],
        },
    }
]

multi_traces = []
ground_truths = []

for q in queries:
    mt = run_search_then_respond(q["query"])
    multi_traces.append(mt)

    # Ground truth uses the responder's expected output
    expected_tools = [
        ToolCall(name=t["name"], arguments=t["arguments"], timestamp=datetime.now(timezone.utc))
        for t in q["expected_tools"].get("responder_agent", [])
    ]
    ground_truths.append(GroundTruth(
        expected_output=q["expected"],
        expected_tool_calls=expected_tools,
    ))

    # Print summary
    search_t = mt.agent_traces["research_agent"]
    resp_t = mt.agent_traces["responder_agent"]
    print(f"\n--- Query: {q['query']}")
    print(f"    Search agent  — messages: {len(search_t.messages)}, tools: {len(search_t.tool_calls)}")
    print(f"    Responder agent — messages: {len(resp_t.messages)}, tools: {len(resp_t.tool_calls)}")
    print(f"    Coordination events: {len(mt.coordination_events)}")
    print(f"    Total latency: {mt.total_latency:.2f}s")

### Evaluate multi-agent traces 

In [ ]:
print(f"\n{'='*50}")
print("MULTI-AGENT STRANDS — EVALUATION RESULTS")
print(f"{'='*50}")

results = []
for i, (mt, gt) in enumerate(zip(multi_traces, ground_truths)):
    result = evaluate(
        trace=mt,  # pass the full MultiAgentTrace
        ground_truth=gt,
        metrics=metrics,
    )

    results.append(result)

    status = "✓" if result.passed else "✗"
    print(f"\n  {status} Query {i+1}: '{queries[i]['query']}'")
    print(f"    Score: {result.overall_score:.2f}")
    for dim in result.dimension_results:
        print(f"      {dim.dimension_name}: {dim.aggregate_score:.2f}")

avg = sum(r.overall_score for r in results) / len(results)
pr = sum(1 for r in results if r.passed) / len(results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")


## Batch Evaluation on Multi Strands Agent

### Load Ground Truth to run batch evaluation 

In [ ]:
import os
import json
import uuid
import pandas as pd
from datetime import datetime, timezone

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)
gt_json = df.to_dict(orient="records")

print(f"✓ Loaded {len(gt_json)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
df.head()


### Query the multi Strands agent created earlier with the GT questions

In [ ]:
# Run each ground truth query through the multi-agent pipeline

multi_traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query = str(row.get("query", row.get("input", row.get("Question", ""))))
    expected = str(row.get("expected_output", row.get("expected", row.get("Answer", ""))))
    context = str(row.get("context", "")) if pd.notna(row.get("context")) else ""

    # Parse expected tool calls (if any in the spreadsheet)
    expected_tools = []
    raw_tools = row.get("expected_tool_calls", row.get("tools", None))
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t.get("name", t.get("tool_name", "")),
                        arguments=t.get("arguments", t.get("parameters", {})),
                        timestamp=datetime.now(timezone.utc)
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    # Run the multi-agent pipeline (research_agent → responder_agent)
    session_id = str(uuid.uuid4())
    mt = run_search_then_respond(query, session_id=session_id)
    multi_traces.append(mt)

    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else []
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    search_t = mt.agent_traces["research_agent"]
    resp_t = mt.agent_traces["responder_agent"]
    print(f"  [{i+1}/{len(gt_json)}] {label}")
    print(f"       research_agent: {len(search_t.messages)} msgs, {len(search_t.tool_calls)} tools")
    print(f"       responder_agent: {len(resp_t.messages)} msgs, {len(resp_t.tool_calls)} tools")
    print(f"       latency: {mt.total_latency:.2f}s")

print(f"\n✓ Ran {len(multi_traces)} queries through the multi-agent pipeline")


### Run batch evaluation on multi Strands agents

In [ ]:
# Batch evaluate the responder traces (the agent producing final answers)
from uaef.evaluation.multi_agent import MultiAgentEvaluator

multi_evaluator = MultiAgentEvaluator(max_workers=4)

responder_traces = [mt.agent_traces["responder_agent"] for mt in multi_traces]

batch_results = multi_evaluator.batch_evaluate(
    traces=multi_traces,
    ground_truths=ground_truths,
)


print(f"{'='*50}")
print(f"BATCH RESULTS — MULTI-AGENT STRANDS")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    query = str(gt_json[i].get("query", gt_json[i].get("input", gt_json[i].get("Question", ""))))
    status = "✓" if r.passed else "✗"
    print(f"\n  {status} Test {i+1}: '{query[:50]}' — Score: {r.overall_score:.2f}")
    for dim in r.dimension_results:
        print(f"      {dim.dimension_name}: {dim.aggregate_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")


### Export batch evaluation results

In [ ]:
# Save multi-agent batch metric results
from uaef.utils import save_metric_results

filepath = save_metric_results(
    batch_results,
    gt_json,
    prefix="strands_multiagent",
    query_key="query",
    expected_key="expected_output",
)


## Previously Deployed Strands Agent

This section is for users who have a previously deployed Strands agent with an invokable HTTPS URL and would like to evaluate it with UAEF.

Instead of creating a local Strands agent, we send requests directly to the deployed endpoint and build an `AgentTrace` from the response.

**Prerequisites:**
- A deployed Strands agent with an HTTPS endpoint
- The endpoint accepts a JSON payload with a `prompt` field and returns a JSON response

### Configure the deployed agent endpoint

In [ ]:
# ── Set your deployed Strands agent URL here ──
DEPLOYED_AGENT_URL = "https://<your-agent-endpoint-url>"
DEPLOYED_AGENT_REGION = "us-east-1" 

# Optional: add headers for authentication (e.g. API key, bearer token)
DEPLOYED_AGENT_HEADERS = {
    "Content-Type": "application/json",
    # "Authorization": "Bearer <your-token>",
}

print(f"✓ Deployed agent URL: {DEPLOYED_AGENT_URL}")

### Helper: invoke the deployed agent

In [ ]:
import json
import time
import urllib.request
import urllib.error
from uuid import uuid4
from datetime import datetime, timezone
from urllib.parse import urlparse

from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.session import Session as BotocoreSession

from uaef.models import AgentTrace, Message, ToolCall
from uaef.models.message import MessageRole


def invoke_deployed_strands_agent(
    url: str,
    user_input: str,
    *,
    region: str = "us-east-1",
    service: str = "lambda",
    session_id: str = None,
) -> AgentTrace:
    """
    Invoke a deployed Strands agent via an IAM-auth Lambda function URL.

    Signs the request with SigV4 using the current AWS credentials
    (from env vars, ~/.aws/credentials, or instance profile).
    """
    session_id = session_id or str(uuid4())

    # ── Build request payload ──
    payload = json.dumps({"prompt": user_input})

    # ── SigV4-sign the request ──
    aws_request = AWSRequest(
        method="POST",
        url=url,
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    # credentials = BotocoreSession().get_credentials().resolve_credentials()
    credentials = BotocoreSession().get_credentials()
    SigV4Auth(credentials, service, region).add_auth(aws_request)

    # ── Send the signed request ──
    req = urllib.request.Request(
        url,
        data=payload.encode("utf-8"),
        headers=dict(aws_request.headers),
        method="POST",
    )

    start = time.time()
    try:
        with urllib.request.urlopen(req) as resp:
            latency = time.time() - start
            body = resp.read().decode("utf-8")
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace") if hasattr(e, "read") else ""
        raise RuntimeError(f"Agent invocation failed ({e.code}): {err_body}")

    # ── Parse response ──
    try:
        parsed = json.loads(body)
        agent_response = (
            parsed.get("response")
            or parsed.get("output")
            or parsed.get("result")
            or parsed.get("answer")
            or parsed.get("text")
            or body
        )
        input_tokens = parsed.get("input_tokens") or parsed.get("usage", {}).get("input_tokens")
        output_tokens = parsed.get("output_tokens") or parsed.get("usage", {}).get("output_tokens")
    except json.JSONDecodeError:
        agent_response = body
        input_tokens = None
        output_tokens = None

    # ── Build canonical AgentTrace ──
    now = datetime.now(timezone.utc)
    messages = [
        Message(role=MessageRole.USER, content=user_input, timestamp=now),
        Message(role=MessageRole.ASSISTANT, content=str(agent_response), timestamp=now),
    ]

    trace = AgentTrace(
        trace_id=str(uuid4()),
        session_id=session_id,
        messages=messages,
        tool_calls=[],
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        latency=latency,
        metadata={"source": "deployed_strands_agent", "url": url},
        framework="strands",
        timestamp=now,
    )
    return trace


print("✓ invoke_deployed_strands_agent() ready")


### Single Query Evaluation

#### Step 1 — Invoke the deployed agent

In [ ]:
user_input = "What's the weather in Seattle?"

agent_trace = invoke_deployed_strands_agent(
    url=DEPLOYED_AGENT_URL,
    user_input=user_input,
    region=DEPLOYED_AGENT_REGION,
)


print(f"✓ Agent responded in {agent_trace.latency:.2f}s")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
if agent_trace.input_tokens:
    print(f"  Input Tokens: {agent_trace.input_tokens}")
if agent_trace.output_tokens:
    print(f"  Output Tokens: {agent_trace.output_tokens}")

for msg in agent_trace.messages:
    print(f"\n  [{msg.role.value}]: {msg.content[:300]}")

#### Step 2 — Select metrics

In [ ]:
metrics = single_ag_metrics
print("You've chosen the following metrics for evaluating the deployed Strands agent:")
for m in metrics:
    print(f"  - {m}")

#### Step 3 — Evaluate

In [ ]:
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall

ground_truth = GroundTruth(
    expected_output="The weather in Seattle is 72°F and sunny",
    expected_tool_calls=[
        ToolCall(
            name="get_weather",
            arguments={"location": "Seattle"},
            timestamp=datetime.now(timezone.utc),
        )
    ],
    context_documents=["Seattle is a city in Washington state"],
)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
)

print(f"\n{'='*50}")
print("DEPLOYED STRANDS AGENT — EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "N/A"
        print(f"  {metric.metric_name}: {score_str}")


### Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation by sending queries from Ground Truth to the deployed Strands agent.

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)
gt_json = df.to_dict(orient="records")

print(f"✓ Loaded {len(gt_json)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

#### Send queries to the deployed Strands agent

In [ ]:
import uuid

traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query = str(row.get("query", row.get("input", row.get("Question", ""))))
    expected = str(row.get("expected_output", row.get("expected", row.get("Answer", ""))))
    context = str(row.get("context", "")) if pd.notna(row.get("context")) else ""

    # Parse expected tool calls
    expected_tools = []
    raw_tools = row.get("expected_tool_calls", row.get("tools", None))
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t.get("name", t.get("tool_name", "")),
                        arguments=t.get("arguments", t.get("parameters", {})),
                        timestamp=datetime.now(timezone.utc),
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    # Invoke the deployed agent
    session_id = str(uuid.uuid4())
    trace = invoke_deployed_strands_agent(
        url=DEPLOYED_AGENT_URL,
        user_input=query,
        session_id=session_id,
    )
    traces.append(trace)

    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else [],
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}  ({trace.latency:.1f}s)")

print(f"\n✓ Ran {len(traces)} queries through the deployed Strands agent")

#### Choose the metrics

In [ ]:
metrics = single_ag_metrics
print("You've chosen the following metrics for evaluating the deployed Strands agent:")
for m in metrics:
    print(f"  - {m}")

#### Run batch evaluation on the agent traces

In [ ]:
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4,
)

print(f"{'='*50}")
print(f"BATCH RESULTS — DEPLOYED STRANDS AGENT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

#### Export batch evaluation results

In [ ]:
# Save deployed agent batch metric results
from uaef.utils import save_metric_results

filepath = save_metric_results(
    batch_results,
    gt_json,
    prefix="strands_deployed",
    query_key="query",
    expected_key="expected_output",
)
